# Train the colony detector (RF-DETR) on a GPU

Runtime → Change runtime type → **GPU** (T4 works; L4/A100 is ~3× faster).

This notebook:
1. downloads the labelled plates from the `dataset-v1` release,
2. builds full-resolution tiles + whole-plate views (test split held out),
3. fine-tunes **RF-DETR Medium** (DINOv2 backbone),
4. evaluates counting accuracy on the held-out test plates,
5. saves `rfdetr_tiles.pth` for the service (`models/`).

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/akashbabu9392/colony_detector.git
%cd colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml

In [ ]:
# Dataset (≈3.2 GB) from the GitHub release
!mkdir -p /content/data && cd /content/data && \
  wget -q https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.zip && \
  wget -q https://github.com/akashbabu9392/colony_detector/releases/download/dataset-v1/data.set.2.zip && \
  unzip -q -o data.set.zip -d raw && unzip -q -o data.set.2.zip -d raw && rm -f *.zip && ls raw

In [ ]:
# Baseline of the training-free engine on the held-out test split
!CD_ENGINES=classical python tools/evaluate.py --dataset /content/data/raw --split test --out /content/eval/classical.csv | tail -25

In [ ]:
# Full-resolution tiles (SCALE=1.0) + one whole-plate view per plate.
SCALE = 1.0
!python tools/build_training_set.py --dataset /content/data/raw --out /content/tiles --scale {SCALE} --empty-keep 0.15 --coco --gold "" | tail -2

In [ ]:
# Fine-tune RF-DETR Medium at 640 px. ~1–3 h on a T4.
!python tools/train_rfdetr.py --dataset /content/tiles/coco --size medium --resolution 640 \
    --epochs 40 --batch 8 --grad-accum 2 --lr 1e-4 --output /content/runs/rfdetr --out models/rfdetr_tiles.pth

In [ ]:
# Counting accuracy on the held-out test plates: RF-DETR alone and with the classical engine
import os
os.environ.update(CD_RFDETR_WEIGHTS="models/rfdetr_tiles.pth", CD_TILE_SCALE=str(SCALE))
!CD_ENGINES=rfdetr python tools/evaluate.py --dataset /content/data/raw --split test --out /content/eval/rfdetr.csv | tail -25
!python tools/tune_fusion.py --dataset /content/data/raw --split valid --engines classical,rfdetr | tail -3
!CD_ENGINES=classical,rfdetr python tools/evaluate.py --dataset /content/data/raw --split test --out /content/eval/ensemble.csv | tail -25

In [ ]:
# Download the trained weights (+ fitted fusion) and put them in the repo's models/ folder
from google.colab import files
files.download("models/rfdetr_tiles.pth")
if os.path.exists("models/fusion.json"): files.download("models/fusion.json")